## Downgrading Python for Compatibility

To ensure compatibility with `tensorflow-cpu==2.8.1` and `piper-phonemize`, we will downgrade the Python version in this Colab environment. TensorFlow 2.8.1 officially supports Python versions up to 3.9. We'll install Python 3.9 using Miniconda.

In [1]:
# Install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
!chmod +x Miniconda3-py39_4.12.0-Linux-x86_64.sh
!bash ./Miniconda3-py39_4.12.0-Linux-x86_64.sh -b -f -p /usr/local

# Add conda to system path
import sys
# Use insert(0, ...) to give Miniconda's site-packages priority
if "/usr/local/lib/python3.9/site-packages" not in sys.path:
    sys.path.insert(0, "/usr/local/lib/python3.9/site-packages")

# Verify conda installation and Python version
!conda init bash
!source ~/.bashrc
!conda activate base
!python --version

print("Python has been set to version 3.9. You may need to restart the runtime (Runtime -> Restart runtime) after this cell for all changes to take effect fully.")
print("After restarting the runtime, re-run all cells from the beginning, but ensure you skip the original Miniconda installation cells if they are present elsewhere in your notebook.")

--2026-07-24 18:12:06--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76607678 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh.2’

Miniconda3-py39_4.1 100%[===================>]  73.06M   319MB/s    in 0.2s    

2026-07-24 18:12:06 (319 MB/s) - ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh.2’ saved [76607678/76607678]

PREFIX=/usr/local
Unpacking payload ...
Solving environment: | / - done

# All requested packages already installed.

installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    dir

In [2]:
import sys
print(f"Python interpreter version: {sys.version}")

Python interpreter version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
print('Checking Python environment after restart:')
!which python
!python --version


Checking Python environment after restart:
/usr/local/bin/python
Python 3.9.12


First, let's install the common build dependencies for Python packages with C extensions, which are crucial for packages like `piper-phonemize`.

In [4]:
# Install common build dependencies for Python packages with C extensions

# Workaround for GLIBCXX_3.4.29 error with apt-get after Miniconda installation
# Temporarily move Miniconda's libstdc++.so.6 to prevent conflict with system apt
import os
miniconda_lib_path = "/usr/local/lib/libstdc++.so.6"
if os.path.exists(miniconda_lib_path):
    print(f"Temporarily moving {miniconda_lib_path}")
    !mv {miniconda_lib_path} {miniconda_lib_path}.bak

!apt-get update
!apt-get install -y build-essential python3-dev

# Restore Miniconda's libstdc++.so.6
if os.path.exists(f"{miniconda_lib_path}.bak"):
    print(f"Restoring {miniconda_lib_path}")
    !mv {miniconda_lib_path}.bak {miniconda_lib_path}

Temporarily moving /usr/local/lib/libstdc++.so.6
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 1s (2,538 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
b

Next, we need to install `libespeak-ng-dev`, which provides the necessary `espeak-ng` headers for `piper-phonemize` to compile. We'll use a workaround to prevent `GLIBCXX` errors that can occur due to environment inconsistencies.

In [5]:
# Install libespeak-ng-dev, a crucial dependency for piper-phonemize's build process
print('Installing libespeak-ng-dev...')

# Workaround for GLIBCXX_3.4.29 error with apt-get after Miniconda installation
# Temporarily move Miniconda's libstdc++.so.6 to prevent conflict with system apt
import os
miniconda_lib_path = "/usr/local/lib/libstdc++.so.6"
if os.path.exists(miniconda_lib_path):
    print(f"Temporarily moving {miniconda_lib_path}")
    !mv {miniconda_lib_path} {miniconda_lib_path}.bak

!env -u LD_LIBRARY_PATH apt-get update
!env -u LD_LIBRARY_PATH apt-get install -y libespeak-ng-dev

# Restore Miniconda's libstdc++.so.6
if os.path.exists(f"{miniconda_lib_path}.bak"):
    print(f"Restoring {miniconda_lib_path}")
    !mv {miniconda_lib_path}.bak {miniconda_lib_path}

Installing libespeak-ng-dev...
Temporarily moving /usr/local/lib/libstdc++.so.6
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libespeak-ng-d

Now that the system dependencies are installed, we will attempt to install `piper-phonemize`. Since direct `pip install git+...` has previously had issues with submodules, we will manually clone the repository, update its submodules, and then install it from the local directory. This ensures all C++ headers, especially `onnxruntime_cxx_api.h`, are correctly present before compilation.

In [6]:
import os
import sys
import subprocess

# --- Step 1: Ensure Python 3.9 from Miniconda is active and persistent ---

# Assuming Miniconda is installed in /usr/local
miniconda_bin_path = '/usr/local/bin'
miniconda_condabin_path = '/usr/local/condabin'
miniconda_python_site_packages = '/usr/local/lib/python3.9/site-packages'

# Add Miniconda paths to the beginning of PATH environment variable for shell commands
if not os.environ['PATH'].startswith(miniconda_bin_path):
    os.environ['PATH'] = miniconda_bin_path + ':' + os.environ['PATH']
if not os.environ['PATH'].startswith(miniconda_condabin_path):
    # This ensures condabin is also considered if it's separate
    os.environ['PATH'] = miniconda_condabin_path + ':' + os.environ['PATH']

# Add Miniconda's site-packages to sys.path for the Python interpreter
if miniconda_python_site_packages not in sys.path:
    sys.path.insert(0, miniconda_python_site_packages)

print("Verifying Python version before piper-phonemize installation...")
!which python
!python --version
print(f"Python interpreter version: {sys.version}")

Verifying Python version before piper-phonemize installation...
/usr/local/bin/python
Python 3.9.12
Python interpreter version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


Finally, we will reinstall `tensorflow-cpu==2.9.0`. This version was identified as compatible with Python 3.9 in a previous step, and re-running the installation ensures it's correctly set up in the current environment.

In [7]:
# Re-install tensorflow-cpu==2.9.0 to ensure correct version for Python 3.9

print("\nRe-installing tensorflow-cpu==2.9.0...")
# Re-attempting installation without interruption.
!pip install tensorflow-cpu==2.9.0


Re-installing tensorflow-cpu==2.9.0...


# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# Upgrade pip to its latest version
!pip install --upgrade pip

# install piper-sample-generator (currently only supports linux systems)
import os

if not os.path.exists('piper-sample-generator'):
    !git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# Re-attempt installing piper-phonemize now that Python 3.9 is active
print("\nAttempting to install piper-phonemize...")
!pip install piper-phonemize

!pip install webrtcvad

# install openwakeword (full installation to support training)
if not os.path.exists('openwakeword'):
    !git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6

# Re-attempt installing tensorflow-cpu==2.8.1 now that Python 3.9 is active
print("\nAttempting to install tensorflow-cpu==2.8.1...")
!pip install tensorflow-cpu==2.8.1

!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

--2026-07-24 18:12:50--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-24T19%3A00%3A52Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-24T18%3A00%3A31Z&ske=2026-07-24T19%3A00%3A52Z&sks=b&skv=2018-11-09&sig=ZtJPVlbciQSmLX39LI3qiu0s%2BpCJv5xTGbasJLx%2BI3E%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NDkyMDM3MSwibmJmIjoxNzg0OTE2NzcxLCJwYXR

In [ ]:
# Re-attempt installing piper-phonemize with maximum verbosity to capture the full build error
print("\nAttempting to install piper-phonemize with verbose output...")
!pip install --verbose --no-cache-dir git+https://github.com/rhasspy/piper-phonemize.git

In [ ]:
# Search for compatible tensorflow-cpu versions for Python 3.9
print("\nSearching for tensorflow-cpu versions compatible with Python 3.9...")
!pip index versions --python-version 3.9 --platform manylinux_2_17_x86_64 tensorflow-cpu

In [ ]:
# Searching for 'speak_lib.h' to resolve the piper-phonemize build error
!find /usr -name speak_lib.h

In [ ]:
# Try to locate the missing 'speak_lib.h' header file
print("Searching for 'speak_lib.h'...")
!find /usr -name speak_lib.h

In [ ]:
# Attempt to install a compatible tensorflow-cpu version
# The previous output showed 2.9.0 as available for Python 3.9.
print("\nAttempting to install tensorflow-cpu==2.9.0...")
!pip install tensorflow-cpu==2.9.0

In [ ]:
# Check the contents of the libespeak-ng-dev package to see where it installs speak_lib.h
print("Checking libespeak-ng-dev package contents...")
!dpkg -L libespeak-ng-dev | grep speak_lib.h

In [ ]:
# Check the contents of the libespeak-ng-dev package to see where it installs speak_lib.h
print("Checking libespeak-ng-dev package contents...")
!dpkg -L libespeak-ng-dev | grep speak_lib.h

In [ ]:
# Check the contents of the libespeak-ng-dev package to see where it installs speak_lib.h
print("Checking libespeak-ng-dev package contents...")
!dpkg -L libespeak-ng-dev | grep speak_lib.h

In [ ]:
# Re-attempt installation of piper-phonemize with verbose output
!pip install --verbose piper-phonemize

In [ ]:
# Re-attempt installation of tensorflow-cpu with verbose output
!pip install --verbose tensorflow-cpu==2.8.1

In [ ]:
# Attempt to install piper-phonemize from a specified extra index URL
# IMPORTANT: Replace `<YOUR_EXTRA_INDEX_URL>` with the actual URL (e.g., a PyPI mirror or a custom repository)
# If you do not have a specific URL, you cannot run this cell as is.
!pip install --verbose piper-phonemize --extra-index-url <YOUR_EXTRA_INDEX_URL>

In [ ]:
# Attempt to install tensorflow-cpu without a specific version constraint
!pip install --verbose tensorflow-cpu

In [ ]:
# Attempt to install the general tensorflow package
!pip install --verbose tensorflow

In [ ]:
# Check the current Python version
import sys
print(sys.version)

In [ ]:
# Check the current Python version
!python --version

In [ ]:
# Check for piper-phonemize availability using --extra-index-url
# Replace <your_extra_index_url> with the actual URL you want to try.
# For example, if you know of a specific PyPI mirror or custom repository, use that URL.
# If you don't have a specific URL, you can try common ones or refer to the package documentation.

# Example (replace with actual URL if known):
# !pip install piper-phonemize --extra-index-url https://pypi.python.org/simple/

print("Please replace '<your_extra_index_url>' with the actual URL you wish to check.")
print("For example: !pip install piper-phonemize --extra-index-url https://download.pytorch.org/whl/cpu")


In [ ]:
# Install piper-phonemize directly from its GitHub repository
!pip install git+https://github.com/rhasspy/piper-phonemize.git

In [ ]:
# Install libespeak-ng-dev, a common dependency for piper-phonemize's build process
!apt-get update
!apt-get install -y libespeak-ng-dev

# Re-attempt installing piper-phonemize directly from its GitHub repository after ensuring espeak-ng-dev is installed
print("\nRe-attempting to install piper-phonemize after installing libespeak-ng-dev...")
!pip install --verbose --no-cache-dir git+https://github.com/rhasspy/piper-phonemize.git

In [ ]:
import os
import sys

# --- Step 1: Ensure Python 3.9 from Miniconda is active and persistent ---

# Assuming Miniconda is installed in /usr/local
miniconda_bin_path = '/usr/local/bin'
miniconda_condabin_path = '/usr/local/condabin'
miniconda_python_site_packages = '/usr/local/lib/python3.9/site-packages'

# Add Miniconda paths to the beginning of PATH environment variable for shell commands
if not os.environ['PATH'].startswith(miniconda_bin_path):
    os.environ['PATH'] = miniconda_bin_path + ':' + os.environ['PATH']
if not os.environ['PATH'].startswith(miniconda_condabin_path):
    # This ensures condabin is also considered if it's separate
    os.environ['PATH'] = miniconda_condabin_path + ':' + os.environ['PATH']

# Add Miniconda's site-packages to sys.path for the Python interpreter
if miniconda_python_site_packages not in sys.path:
    sys.path.insert(0, miniconda_python_site_packages)

print("Verifying Python version after path update...")
!which python
!python --version
print(f"Python interpreter version: {sys.version}")

# --- Step 2: Manually clone piper-phonemize and initialize submodules ---

print("\nManually cloning piper-phonemize and initializing submodules...")
piper_phonemize_src_path = "/content/piper-phonemize-src"

# Remove existing directory if it exists to ensure a clean clone
if os.path.exists(piper_phonemize_src_path):
    print(f"'{piper_phonemize_src_path}' already exists, removing...")
    !rm -rf {piper_phonemize_src_path}

!git clone https://github.com/rhasspy/piper-phonemize.git {piper_phonemize_src_path}
!cd {piper_phonemize_src_path} && git submodule update --init --recursive

# --- Step 3: Verify if necessary headers are present after submodule update ---

print("\nVerifying onnxruntime_cxx_api.h presence after submodule update...")
!ls {piper_phonemize_src_path}/lib/Linux-x86_64/onnxruntime/include/onnxruntime_cxx_api.h

print("\nVerifying espeak-ng/speak_lib.h presence after submodule update...")
!ls {piper_phonemize_src_path}/espeak-ng/build/include/espeak-ng/speak_lib.h

# --- Step 4: Re-attempt installing piper-phonemize from the local cloned directory ---

print("\nRe-attempting to install piper-phonemize from local clone...")
# Use --break-system-packages if pip warns about outside virtual environment
!pip install --verbose --no-cache-dir {piper_phonemize_src_path}

# --- Step 5: Re-install tensorflow-cpu==2.9.0 to ensure correct version for Python 3.9 ---

print("\nRe-installing tensorflow-cpu==2.9.0...")
!pip install tensorflow-cpu==2.9.0

Let's manually verify the submodule status and then re-run the update command to ensure all necessary files are downloaded.

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Checking contents of {piper_phonemize_src_path}/espeak-ng/")
!ls -la {piper_phonemize_src_path}/espeak-ng/

print(f"\nChecking contents of {piper_phonemize_src_path}/lib/Linux-x86_64/onnxruntime/")
!ls -la {piper_phonemize_src_path}/lib/Linux-x86_64/onnxruntime/

print("\nRe-running git submodule update...")
!cd {piper_phonemize_src_path} && git submodule update --init --recursive

print("\nVerifying onnxruntime_cxx_api.h presence after re-running submodule update...")
!ls {piper_phonemize_src_path}/lib/Linux-x86_64/onnxruntime/include/onnxruntime_cxx_api.h

print("\nVerifying espeak-ng/speak_lib.h presence after re-running submodule update...")
!ls {piper_phonemize_src_path}/espeak-ng/build/include/espeak-ng/speak_lib.h

It appears the `piper-phonemize` repository was not fully cloned or its submodules were not initialized. Let's ensure a clean clone and submodule update before proceeding.

In [ ]:
import os
import subprocess
import shutil

piper_phonemize_src_path = "/content/piper-phonemize-src"
espeak_ng_src_path = os.path.join(piper_phonemize_src_path, "espeak-ng")

# Ensure clean clone of piper-phonemize
print(f"Ensuring '{piper_phonemize_src_path}' is cleanly cloned...")

# Remove existing directory to ensure a fresh start
if os.path.exists(piper_phonemize_src_path):
    print(f"Removing existing directory: {piper_phonemize_src_path}")
    shutil.rmtree(piper_phonemize_src_path)

# Clone the piper-phonemize repository
print("Cloning piper-phonemize repository...")
subprocess.run(["git", "clone", "https://github.com/rhasspy/piper-phonemize.git", piper_phonemize_src_path], check=True, capture_output=True, text=True)

# --- BEGIN NEW FIX: Manually clone espeak-ng into the submodule path ---
print(f"Manually cloning eSpeak-ng into submodule directory: {espeak_ng_src_path}...")
if os.path.exists(espeak_ng_src_path):
    print(f"Removing existing eSpeak-ng directory: {espeak_ng_src_path}")
    shutil.rmtree(espeak_ng_src_path)
subprocess.run(["git", "clone", "https://github.com/rhasspy/espeak-ng.git", espeak_ng_src_path], check=True, capture_output=True, text=True)
print("eSpeak-ng repository cloned successfully.")
# --- END NEW FIX ---

# Explicitly list contents to verify
print(f"DEBUG: Listing contents of {piper_phonemize_src_path} recursively after manual eSpeak-ng clone:")
# Change to capture_output=True and print it to ensure visibility in Colab output
ls_result = subprocess.run(["ls", "-laR", piper_phonemize_src_path], check=False, capture_output=True, text=True)
print("LS -laR STDOUT:\n" + ls_result.stdout)
if ls_result.stderr:
    print("LS -laR STDERR:\n" + ls_result.stderr)


# --- BEGIN FIX: Add explicit checks for espeak-ng submodule population (kept from previous iteration) ---
print(f"Verifying contents of espeak-ng submodule directory ({espeak_ng_src_path})...")
if not os.path.isdir(espeak_ng_src_path) or not os.listdir(espeak_ng_src_path):
    raise RuntimeError(
        f"Error: eSpeak-ng submodule directory '{espeak_ng_src_path}' is missing or empty after 'git clone'.\n"
        "This indicates a persistent problem with the eSpeak-ng cloning process."
    )

espeak_ng_configure_ac_path = os.path.join(espeak_ng_src_path, "configure.ac")
if not os.path.exists(espeak_ng_configure_ac_path):
    print(f"DEBUG: Listing contents of {espeak_ng_src_path}:")
    ls_espeak_ng_result = subprocess.run(["ls", "-l", espeak_ng_src_path], check=False, capture_output=True, text=True)
    print("LS -l eSpeak-ng STDOUT:\n" + ls_espeak_ng_result.stdout)
    if ls_espeak_ng_result.stderr:
        print("LS -l eSpeak-ng STDERR:\n" + ls_espeak_ng_result.stderr)
    raise RuntimeError(
        f"Error: 'configure.ac' not found in eSpeak-ng submodule directory '{espeak_ng_src_path}' after clone.\n"
        "This means the eSpeak-ng source code is incomplete. The `git clone` command failed to retrieve all necessary files."
    )

print(f"'configure.ac' found in {espeak_ng_src_path}. eSpeak-ng submodule appears properly populated.")
# --- END FIX ---

print("Please re-run the `espeak-ng` build cell (8d3b4887) next, followed by the installation cell (cf154c58).")

Now that we've explicitly re-initialized the submodules and verified the presence of the header files, we can re-attempt the installation of `piper-phonemize` from the local cloned directory.

In [ ]:
import os
import shutil

# Ensure the current working directory is always a stable, existing path (e.g., /content)
# This prevents FileNotFoundError from os.getcwd() if a previous operation (like rmtree)
# inadvertently removed the current working directory.
os.chdir("/content")

piper_phonemize_src_path = "/content/piper-phonemize-src"
espeak_ng_install_path = os.path.join(piper_phonemize_src_path, "espeak-ng/install") # Must match path from 8d3b4887
onnxruntime_install_path = os.path.join(piper_phonemize_src_path, "lib/Linux-x86_64/onnxruntime")

# Set environment variables for piper-phonemize's setup.py to find espeak-ng and onnxruntime
# This is crucial for the build process to locate the headers and libraries.
os.environ['ESPEAK_NG_INCLUDE_DIR'] = os.path.join(espeak_ng_install_path, "include")
os.environ['ESPEAK_NG_LIB_DIR'] = os.path.join(espeak_ng_install_path, "lib")
os.environ['ONNXRUNTIME_INCLUDE_DIR'] = os.path.join(onnxruntime_install_path, "include")
os.environ['ONNXRUNTIME_LIB_DIR'] = os.path.join(onnxruntime_install_path, "lib")

print(f"ESPEAK_NG_INCLUDE_DIR set to: {os.environ['ESPEAK_NG_INCLUDE_DIR']}")
print(f"ESPEAK_NG_LIB_DIR set to: {os.environ['ESPEAK_NG_LIB_DIR']}")
print(f"ONNXRUNTIME_INCLUDE_DIR set to: {os.environ['ONNXRUNTIME_INCLUDE_DIR']}")
print(f"ONNXRUNTIME_LIB_DIR set to: {os.environ['ONNXRUNTIME_LIB_DIR']}")

print("\nRe-attempting to install piper-phonemize from local clone with configured eSpeak-ng and ONNX Runtime paths...")
# Use --break-system-packages if pip warns about outside virtual environment
# Use --verbose to see detailed build output
!pip install --verbose --no-cache-dir {piper_phonemize_src_path}

# --- Re-install tensorflow-cpu==2.9.0 to ensure correct version for Python 3.9 --- to be sure

print("\nRe-installing tensorflow-cpu==2.9.0 to ensure consistency...")
!pip install tensorflow-cpu==2.9.0

In [ ]:
import sys

# Ensure Miniconda's Python 3.9 site-packages are in sys.path for the current kernel
miniconda_python_site_packages = '/usr/local/lib/python3.9/site-packages'
if miniconda_python_site_packages not in sys.path:
    sys.path.insert(0, miniconda_python_site_packages)

print('Verifying piper_phonemize installation...')
print(f"Current Python interpreter version: {sys.version}")
print(f"Effective sys.path: {sys.path}")

import piper_phonemize

# Test a simple phonemization
text = "hello world"
phonemes = piper_phonemize.phonemize([text], language='en-us')

print(f"Phonemes for '{text}': {phonemes}")
print("piper_phonemize imported and tested successfully!")

In [ ]:
import os

python39_site_packages = '/usr/local/lib/python3.9/site-packages'
print(f"Listing contents of {python39_site_packages} related to piper_phonemize...")

# Look for the .so file that should be the C++ extension
!ls -l {python39_site_packages}/piper_phonemize_cpp*

# Also list the contents of the piper_phonemize package directory
!ls -l {python39_site_packages}/piper_phonemize/

print("Please review the output to ensure 'piper_phonemize_cpp' shared library is present.")

In [ ]:
import os
import subprocess
import shutil

# Ensure the current working directory is always a stable, existing path (e.g., /content)
# This prevents FileNotFoundError from os.getcwd() if a previous operation (like rmtree)
# inadvertently removed the current working directory.
os.chdir("/content")


pip_env = os.environ.copy()
# Unset LD_LIBRARY_PATH for the subprocess calls to prevent conflicts with apt-get and system libraries
# This ensures that apt-get and other system tools use their default libraries.
if 'LD_LIBRARY_PATH' in pip_env:
    del pip_env['LD_LIBRARY_PATH']

piper_phonemize_src_path = "/content/piper-phonemize-src"
espeak_ng_src_path = os.path.join(piper_phonemize_src_path, "espeak-ng")
espeak_ng_install_path = os.path.join(espeak_ng_src_path, "install") # Custom install prefix for autotools

print(f"Ensuring eSpeak-ng build dependencies are installed...")

# --- GLIBCXX_3.4.29 FIX: Temporarily move Miniconda's libstdc++.so.6 for the *entire* eSpeak-ng build process ---
miniconda_lib_path = "/usr/local/lib/libstdc++.so.6"
miniconda_lib_bak_path = f"{miniconda_lib_path}.bak"

# Move Miniconda's libstdc++.so.6 out of the way if it exists
if os.path.exists(miniconda_lib_path):
    print(f"Temporarily moving {miniconda_lib_path} to {miniconda_lib_bak_path} for eSpeak-ng build.")
    subprocess.run(["mv", miniconda_lib_path, miniconda_lib_bak_path], check=False, capture_output=True)


print("Updating apt-get and installing eSpeak-ng build dependencies...")
subprocess.run(["apt-get", "update"], env=pip_env, check=True, capture_output=True)
# Install autotools and other common espeak-ng build dependencies
install_cmd = [
    "apt-get", "install", "-y",
    "autotools-dev", "automake", "libtool", "pkg-config", "flex", "libsndfile1-dev"
]
subprocess.run(install_cmd, env=dict(pip_env, DEBIAN_FRONTEND="noninteractive"), check=True, capture_output=True)

print(f"Building eSpeak-ng submodule using autotools in {espeak_ng_src_path}...")

# Clean previous build artifacts and install directory
if os.path.exists(espeak_ng_install_path):
    print(f"Removing existing install directory: {espeak_ng_install_path}")
    shutil.rmtree(espeak_ng_install_path)

# espeak-ng also creates a build directory, remove that too for clean state
if os.path.exists(os.path.join(espeak_ng_src_path, "build")):
    print(f"Removing existing build directory: {os.path.join(espeak_ng_src_path, 'build')}")
    shutil.rmtree(os.path.join(espeak_ng_src_path, "build"))

# Change to the espeak-ng source directory
original_cwd = os.getcwd()
os.chdir(espeak_ng_src_path)

print(f"DEBUG: Listing contents of eSpeak-ng source directory ({os.getcwd()}):")
# Add check=True to make sure this command's failure is caught
subprocess.run(["ls", "-l"], check=True, capture_output=True, text=True)

try:
    # Create dummy AUTHORS and NEWS files if they don't exist, as automake sometimes expects them
    if not os.path.exists("AUTHORS"):
        print("Creating dummy AUTHORS file...")
        with open("AUTHORS", "w") as f:
            f.write("Dummy AUTHORS file for build process\n")
    if not os.path.exists("NEWS"):
        print("Creating dummy NEWS file...")
        with open("NEWS", "w") as f:
            f.write("Dummy NEWS file for build process\n")

    # Run autogen.sh or autoreconf -i to generate configure script
    print("Attempting to generate configure script using autoreconf -i for eSpeak-ng...")
    subprocess.run(["autoreconf", "-i"], check=True, capture_output=True, text=True)

    # Run configure with a custom prefix for installation
    print(f"Running ./configure --prefix={espeak_ng_install_path} for eSpeak-ng...")
    subprocess.run(["./configure", f"--prefix={espeak_ng_install_path}"], check=True, capture_output=True, text=True)

    # Get the number of CPU cores for parallel compilation
    nproc_result = subprocess.run(["nproc"], capture_output=True, text=True, check=True)
    num_cores = nproc_result.stdout.strip()
    print(f"Detected {num_cores} CPU cores.")

    # Run make to compile the source
    print("Running make for eSpeak-ng...")
    subprocess.run(["make", f"-j{num_cores}"], check=True, capture_output=True, text=True)

    # Run make install to place headers and libs in the specified prefix
    print(f"Running make install to {espeak_ng_install_path} for eSpeak-ng...")
    subprocess.run(["make", "install"], check=True, capture_output=True, text=True)

except subprocess.CalledProcessError as e:
    print(f"Error building eSpeak-ng: {e}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")
    os.chdir(original_cwd) # Change back before raising
    raise # Re-raise the exception to stop execution
finally:
    os.chdir(original_cwd)

    # --- GLIBCXX_3.4.29 FIX: Restore Miniconda's libstdc++.so.6 after eSpeak-ng build ----
    if os.path.exists(miniconda_lib_bak_path):
        print(f"Restoring {miniconda_lib_path} from backup.")
        subprocess.run(["mv", miniconda_lib_bak_path, miniconda_lib_path], check=False, capture_output=True)

print("eSpeak-ng submodule built and installed successfully.")

# Verify speak_lib.h presence after building and installing espeak-ng (corrected verification)
espeak_ng_header_path = os.path.join(espeak_ng_install_path, "include/espeak-ng/speak_lib.h")
print(f"Verifying speak_lib.h presence at: {espeak_ng_header_path}")
ls_result = subprocess.run(["ls", espeak_ng_header_path], capture_output=True, text=True)
if ls_result.returncode == 0:
    print(ls_result.stdout)
else:
    print(f"Error verifying speak_lib.h: {ls_result.stderr}")

# Add recursive listing of the installed include directory to inspect what was actually installed
print(f"\nRecursive listing of installed eSpeak-ng include directory ({espeak_ng_install_path}/include):")
ls_include_result = subprocess.run(["ls", "-R", os.path.join(espeak_ng_install_path, "include")], capture_output=True, text=True)
print(ls_include_result.stdout)
if ls_include_result.stderr:
    print(f"LS INCLUDE STDERR:\n{ls_include_result.stderr}")

print("\nNow, let's configure the environment for piper-phonemize installation.")

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"
phonemize_cpp_path = os.path.join(piper_phonemize_src_path, "src", "phonemize.cpp")

print(f"Reading original content of {phonemize_cpp_path}...")
with open(phonemize_cpp_path, 'r') as f:
    content = f.readlines()

# Remove the line '#include <espeak-ng/speak_api.h>' if it exists
original_content = list(content)
content = [line for line in content if "#include <espeak-ng/speak_api.h>" not in line]

if len(original_content) != len(content):
    print(f"Removed '#include <espeak-ng/speak_api.h>' from {phonemize_cpp_path}...")
    with open(phonemize_cpp_path, 'w') as f:
        f.writelines(content)
    print("File modified successfully.")
else:
    print("'#include <espeak-ng/speak_api.h>' not found or already removed. No modification needed.")

# Ensure '#include <espeak-ng/speak_lib.h>' is present (it should be by default)
if "#include <espeak-ng/speak_lib.h>\n" not in content:
    print(f"Warning: '#include <espeak-ng/speak_lib.h>' not found. Attempting to add it...")
    # Find a suitable line to insert the include, e.g., after other includes
    insert_index = -1
    for i, line in enumerate(content):
        if line.strip().startswith('#include'):
            insert_index = i + 1
        else:
            break
    if insert_index != -1:
        content.insert(insert_index, "#include <espeak-ng/speak_lib.h>\n")
    else:
        content.insert(0, "#include <espeak-ng/speak_lib.h>\n") # Fallback at beginning
    with open(phonemize_cpp_path, 'w') as f:
        f.writelines(content)
    print("Added '#include <espeak-ng/speak_lib.h>'.")

print("\nPlease re-run cell 'cf154c58' to try installing piper-phonemize again with the corrected source.")

In [ ]:
import os
import shutil

piper_phonemize_src_path = "/content/piper-phonemize-src"
espeak_ng_src_path = os.path.join(piper_phonemize_src_path, "espeak-ng")
espeak_ng_install_include_path = os.path.join(espeak_ng_src_path, "install", "include")
espeak_ng_build_include_path = os.path.join(espeak_ng_src_path, "build", "include")

# Clean up existing symlinks or directories in the build directory to avoid conflicts
print("Cleaning up old espeak-ng symlinks/directories in build directory...")
old_symlink_target_dir = os.path.join(espeak_ng_build_include_path, "espeak-ng")
if os.path.islink(old_symlink_target_dir):
    print(f"Removing old symlink: {old_symlink_target_dir}")
    os.remove(old_symlink_target_dir)
elif os.path.isdir(old_symlink_target_dir):
    # If it's a directory (e.g., from an incomplete prior operation), remove it
    print(f"Removing old directory: {old_symlink_target_dir}")
    shutil.rmtree(old_symlink_target_dir)

# Ensure the parent directory for the symlink destination exists
os.makedirs(espeak_ng_build_include_path, exist_ok=True)

# Create the new symlink: espeak-ng/build/include/espeak-ng -> espeak-ng/install/include/espeak-ng
# This ensures that if g++ implicitly searches in 'espeak-ng/build/include', it finds the correct headers
source_for_symlink = os.path.join(espeak_ng_install_include_path, "espeak-ng")
destination_for_symlink = os.path.join(espeak_ng_build_include_path, "espeak-ng")

if not os.path.exists(source_for_symlink):
    print(f"Error: Source for symlink does not exist: {source_for_symlink}")
    print("This indicates an issue with the eSpeak-ng manual build or install process.")
else:
    if not os.path.exists(destination_for_symlink):
        print(f"Creating symlink: {destination_for_symlink} -> {source_for_symlink}")
        os.symlink(source_for_symlink, destination_for_symlink)
    else:
        print(f"Symlink/directory already exists at {destination_for_symlink}, skipping creation.")

print("\nVerifying speak_lib.h presence via the new build path:")
!ls {destination_for_symlink}/speak_lib.h

print("eSpeak-ng headers should now be accessible to piper-phonemize's build process via the expected build directory path.")
print("Please re-run cell 'cf154c58' to try installing piper-phonemize again.")

In [ ]:
print(f"Searching for 'onnxruntime_cxx_api.h' in {piper_phonemize_src_path}...")
!find {piper_phonemize_src_path} -name onnxruntime_cxx_api.h

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Checking git submodule status in {piper_phonemize_src_path}...")
!cd {piper_phonemize_src_path} && git submodule status

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Listing contents of {piper_phonemize_src_path}/lib recursively...")
!ls -R {piper_phonemize_src_path}/lib

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Searching for 'speak_lib.h' within {piper_phonemize_src_path}...")
!find {piper_phonemize_src_path} -name speak_lib.h

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"
espeak_ng_target_dir = os.path.join(piper_phonemize_src_path, "espeak-ng/build/include")
espeak_ng_link_path = os.path.join(espeak_ng_target_dir, "espeak-ng")
espeak_ng_source_path = "/usr/include/espeak-ng"

print(f"Creating directory: {espeak_ng_target_dir}")
!mkdir -p {espeak_ng_target_dir}

if not os.path.exists(espeak_ng_link_path):
    print(f"Creating symbolic link: {espeak_ng_source_path} -> {espeak_ng_link_path}")
    !ln -s {espeak_ng_source_path} {espeak_ng_link_path}
else:
    print(f"Symbolic link already exists: {espeak_ng_link_path}")

print("Verifying speak_lib.h presence via the new link path:")
!ls {espeak_ng_link_path}/speak_lib.h

print("eSpeak-ng headers should now be accessible to piper-phonemize's build process.")
print("Please re-run cell 'cf154c58' to try installing piper-phonemize again.")

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"
onnxruntime_dir = os.path.join(piper_phonemize_src_path, "lib/Linux-x86_64/onnxruntime")

print(f"Listing contents of {onnxruntime_dir}/include/ to verify header accessibility...")
!ls {onnxruntime_dir}/include/

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Listing contents of {piper_phonemize_src_path}/src/ to verify file paths...")
!ls -l {piper_phonemize_src_path}/src/

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"

print(f"Listing contents of {piper_phonemize_src_path} recursively...")
!ls -R {piper_phonemize_src_path}

In [ ]:
import os

piper_phonemize_src_path = "/content/piper-phonemize-src"
onnxruntime_version = "1.15.1" # This version is often specified in piper-phonemize's Makefile
onnxruntime_dir = os.path.join(piper_phonemize_src_path, "lib/Linux-x86_64/onnxruntime")

print(f"Manually downloading and extracting ONNX Runtime C++ library (version {onnxruntime_version})...")

# Create the target directory if it doesn't exist
!mkdir -p {onnxruntime_dir}

# Download and extract ONNX Runtime C++ library
!curl -sSL https://github.com/microsoft/onnxruntime/releases/download/v{onnxruntime_version}/onnxruntime-linux-x64-{onnxruntime_version}.tgz | tar xz -C {onnxruntime_dir} --strip-components=1

# Ensure correct permissions for the extracted ONNX Runtime files
print(f"Setting permissions for ONNX Runtime directory: {onnxruntime_dir}")
!chmod -R 755 {onnxruntime_dir}

print("ONNX Runtime C++ library downloaded and extracted.")

# Verify its presence
print("Verifying onnxruntime_cxx_api.h presence:")
!ls {onnxruntime_dir}/include/onnxruntime_cxx_api.h

print("Now that ONNX Runtime C++ headers are in place, please re-run cell 'cf154c58' to try installing piper-phonemize again.")

In [ ]:
# Re-attempt installing piper-phonemize directly from its GitHub repository after installing espeak-ng-tools
!pip install git+https://github.com/rhasspy/piper-phonemize.git

In [ ]:
# Install common build dependencies for Python packages with C extensions
!apt-get update
!apt-get install -y build-essential python3-dev

In [ ]:
# Re-attempt installing piper-phonemize directly from its GitHub repository after installing core build tools
!pip install git+https://github.com/rhasspy/piper-phonemize.git

In [ ]:
# Install Rust toolchain (rustc and cargo) for Rust-based Python packages
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ':/root/.cargo/bin'

In [ ]:
# Re-attempt installing piper-phonemize directly from its GitHub repository after installing Rust
!pip install git+https://github.com/rhasspy/piper-phonemize.git

### How to Search for Compatible Wheels on PyPI

To find pre-compiled wheels (`.whl` files) compatible with your Python version (e.g., `cp312` for Python 3.12 or `cp314` for Python 3.14), you'll need to visit the project pages on PyPI (Python Package Index).

1.  **Go to PyPI for `piper-phonemize`**: Navigate to [https://pypi.org/project/piper-phonemize/#files](https://pypi.org/project/piper-phonemize/#files)
2.  **Go to PyPI for `tensorflow`**: Navigate to [https://pypi.org/project/tensorflow/#files](https://pypi.org/project/tensorflow/#files) (Note: `tensorflow-cpu` is often just `tensorflow` for specific builds, and newer `tensorflow` versions might include CPU-only variants or support your Python).
3.  **Look for `.whl` files**: On each page, scroll down to the "Download files" section.
4.  **Identify compatible wheels**: Look for filenames that contain `cp312` (for CPython 3.12) or `cp314` (for CPython 3.14) and `manylinux` or `linux_x86_64` (for Linux 64-bit systems, which Colab uses). For example, a wheel for Python 3.12 might look like `piper_phonemize-X.Y.Z-cp312-cp312-manylinux_2_17_x86_64.whl`.
5.  **Copy the direct download URL**: Right-click on the compatible `.whl` file and copy the link address.

### Install a Found Wheel Directly (Example)

If you find a compatible wheel for either package, you can try installing it directly using its URL or by downloading it and installing locally. Here's how to try installing it from a direct URL:

In [ ]:
# Replace 'YOUR_WHEEL_URL_HERE' with the direct download link of the compatible .whl file you found on PyPI
# Example for piper-phonemize:
# !pip install https://files.pythonhosted.org/packages/.../piper_phonemize-X.Y.Z-cp312-cp312-manylinux_2_17_x86_64.whl

# Example for tensorflow (ensure it's a version compatible with your Python):
# !pip install https://files.pythonhosted.org/packages/.../tensorflow-X.Y.Z-cp312-cp312-manylinux_2_17_x86_64.whl

# To install piper-phonemize if you find a wheel:
# !pip install YOUR_PIPER_PHONEMIZE_WHEEL_URL_HERE

# To install tensorflow (or tensorflow-cpu) if you find a wheel:
# !pip install YOUR_TENSORFLOW_WHEEL_URL_HERE

# If you don't find specific `cp312` or `cp314` wheels, try installing without a version constraint for TensorFlow
# and see if a newer version is picked up that supports your Python.
# Note: This is less likely to work for `piper-phonemize` if it requires compilation and a specific Python API.

print("Please update this cell with the actual wheel URLs you find.")

In [ ]:
import sys

# Install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Update path
sys.path.append('/usr/local/lib/python3.10/site-packages')

# Verify conda installation
!conda --version

In [ ]:
# Accept Conda Terms of Service
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Search for piper-phonemize in conda channels
!conda search piper-phonemize

In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["morpheus"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

In [ ]:
import sys
print(sys.version)
!python --version

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!

In [ ]:
import os
import yaml

# Ensure config is defined. If not, reload it from 'my_model.yaml'
if 'config' not in locals():
    try:
        with open('my_model.yaml', 'r') as file:
            config = yaml.load(file, yaml.Loader)
        print("Config reloaded from 'my_model.yaml'.")
    except FileNotFoundError:
        print("Error: 'my_model.yaml' not found. Please ensure it was created by running cell 482cf2d0.")
        config = {'model_name': 'morpheus'} # Fallback to a default if file not found
    except Exception as e:
        print(f"Error loading config from 'my_model.yaml': {e}")
        config = {'model_name': 'morpheus'} # Fallback to a default on other errors

model_name = config.get('model_name', 'morpheus')
model_output_dir = f"my_custom_model"

print(f"Verifying model files in: {model_output_dir}")

if os.path.exists(model_output_dir):
    !ls -l {model_output_dir}

    onnx_file = os.path.join(model_output_dir, f"{model_name}.onnx")
    tflite_file = os.path.join(model_output_dir, f"{model_name}.tflite")

    if os.path.exists(onnx_file):
        print(f"\nFound ONNX model: {onnx_file}")
    else:
        print(f"\nONNX model NOT found: {onnx_file}")

    if os.path.exists(tflite_file):
        print(f"Found TFLite model: {tflite_file}")
    else:
        print(f"TFLite model NOT found: {tflite_file}")
else:
    print(f"Output directory '{model_output_dir}' does not exist. Model conversion might not have completed successfully or the directory path is incorrect.")


In [ ]:
import os
import yaml

# Ensure config is defined. If not, reload it from 'my_model.yaml'
if 'config' not in locals() or config.get('model_name') == 'morpheus':
    try:
        with open('my_model.yaml', 'r') as file:
            config = yaml.load(file, yaml.Loader)
        print("Config reloaded from 'my_model.yaml'.")
    except FileNotFoundError:
        print("Error: 'my_model.yaml' not found. Please ensure it was created by running cell 482cf2d0.")
        config = {'model_name': 'morpheus'} # Fallback to a default if file not found
    except Exception as e:
        print(f"Error loading config from 'my_model.yaml': {e}")
        config = {'model_name': 'morpheus'} # Fallback to a default on other errors

model_name = config.get('model_name', 'morpheus')
model_output_dir = f"my_custom_model"

print(f"Verifying model files in: {model_output_dir}")

if os.path.exists(model_output_dir):
    print(f"Contents of {model_output_dir}:")
    !ls -l {model_output_dir}

    onnx_file = os.path.join(model_output_dir, f"{model_name}.onnx")
    tflite_file = os.path.join(model_output_dir, f"{model_name}.tflite")

    if os.path.exists(onnx_file):
        print(f"\nFound ONNX model: {onnx_file}")
    else:
        print(f"\nONNX model NOT found: {onnx_file}")

    if os.path.exists(tflite_file):
        print(f"Found TFLite model: {tflite_file}")
    else:
        print(f"TFLite model NOT found: {tflite_file}")
else:
    print(f"Output directory '{model_output_dir}' does not exist. Model conversion might not have completed successfully or the directory path is incorrect.")

In [ ]:
print('Installing libespeak-ng-dev...')
!env -u LD_LIBRARY_PATH apt-get update
!env -u LD_LIBRARY_PATH apt-get install -y libespeak-ng-dev